# 24. Game model, season engine, and a backtest against real standings (simulator step 2)

Notebook 23 gave every team a lineup and a strength (WAR per game). This notebook turns lineups into **games, points and standings**, and then asks the question that matters: **does it predict real standings better than a simple rule?**

**The game model** (code: `src/simulate.py`):

1. **Normal goals.** Each team's goals, except late pulled-goalie goals, are random (Poisson) draws.
   - Their total is the league's scoring level.
   - The expected difference is `home ice + a_s x (skater WAR per game, home - away) + a_g x (goalie WAR per game, home - away)`.
2. **The late game.** With the goalie pulled, a trailing team sometimes ties it, and a leading team often scores into the empty net. This comes from a table of what really happened.
3. **Overtime or a shootout** if still tied. The home team wins it with a probability that tilts a little with strength. Win = 2 points, OT/SO loss = 1.

**What's fit** on preseason projected lineup WAR (what the simulator will use): home ice, a_s, a_g, the scoring level, and the overtime tilt.

**Two lineup fixes from notebook 23:**

- minutes blended with each player's own ice time (McDavid's slot minutes were too low);
- call-ups only when the NHL level can't fill a slot.

**The main test:** every past season, simulated 1,000 times from what was known before it, against the real standings, and against two simple rules. Our model has to beat "last season's points, pulled one-third toward average". The verdict is reported plainly either way.

**Player randomness:** each player's season can be better or worse than projected (notebook 17's draws). Which version to use is picked here, by whether the simulated standings spread out as much as real ones.

Data: past results 2008-09 to 2020-21 from the NHL API (`data/schedule_raw/`, downloaded once, gitignored); 2021-22 to 2025-26 from the shot data.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skellam

from forecast import history_table, project, backtest, aging_curve, group_average, current_projections, add_age, GOALS_PER_WIN
from edge_data import build_bio_table, download_club_schedules, schedule_games, nhl_code
from nhle import load_profiles
from development import (training_nhle, best_nhle, skater_features, fit_prior, apply_prior, candidates, fit_rows,
                         experience, TARGETS, AGING, SK_WEIGHTS, SK_REG, G_WEIGHTS, G_REG, PRIOR_INPUTS)
import lineups as L
import simulate as S

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

NAMES = {"F": "Forwards", "D": "Defensemen", "G": "Goalies"}

skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
history = history_table(skater_war, goalie_war)
skaters, goalies = history[history["group"] != "G"], history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]

# Notebook 23's choices (picked by its backtest)
CHOSEN = {"F": (0.75, "R1"), "D": (0.75, "R1"), "G": (0.5, "R1")}    # coach score: w and the rookie rule
LOOKBACK = {"F": 2, "D": 1, "G": 2}                                  # seasons looked back for usage
N_SIMS, SEED = 1000, 24
print("Train targets:", ", ".join(label(t) for t in TRAIN))
print("Test targets: ", ", ".join(label(t) for t in TEST))

Train targets: 2009-10, 2010-11, 2011-12, 2013-14, 2014-15, 2015-16, 2016-17, 2017-18
Test targets:  2018-19, 2021-22, 2022-23, 2023-24, 2024-25, 2025-26


## 2. Real results

- **2008-09 to 2020-21:** the NHL API's season schedule for each team, with final scores and how each game ended. These were downloaded once; the call below only fetches files that are missing, so a rerun makes no network calls.
- **2021-22 to 2025-26:** the shot data. It also shows *which kind* of goal each one was (section 3).

Points are counted from the games: win 2, OT/SO loss 1. In the standings a shootout winner gets one goal, so that's counted too.

In [2]:
team_seasons = sorted({(nhl_code(t, s), s) for t, s in zip(skater_war["team"], skater_war["season"]) if s <= 2020})
failed = download_club_schedules(team_seasons, DATA / "schedule_raw")
assert not failed
old = schedule_games(DATA / "schedule_raw")
recent = S.games_from_shots(pd.read_parquet(DATA / "shots_2021-2025.parquet"))
games = pd.concat([old, recent], ignore_index=True)
results = S.add_results(games)
results["reg_goals"] = results["home_goals"] + results["away_goals"] - results["ot"]   # the OT/SO winner's goal off
real = S.standings(games)

per = results.groupby("season").agg(games=("game_id", "size"), goals_per_game=("reg_goals", "mean"),
                                    home_win=("home_win", "mean"), ot_so=("ot", "mean"),
                                    shootout=("last_period", lambda x: (x == "SO").mean()))
per["games per team"] = real.groupby("season")["gp"].agg(lambda x: f"{x.min()}-{x.max()}" if x.min() != x.max() else str(x.min()))
per.index = per.index.map(label)
print("Regular-season games (goals per game = regulation goals):")
print(per.round(3).to_string())
print(f"\n2021-22: {len(recent[recent['season'] == 2021])} games (one game has no shot records, so it's missing)")

394 already saved, 0 to download (about 0 minutes)
Done, no failures.
Regular-season games (goals per game = regulation goals):
         games  goals_per_game  home_win  ot_so  shootout games per team
season                                                                  
2008-09   1230           5.596     0.559  0.229     0.129             82
2009-10   1230           5.436     0.561  0.245     0.150             82
2010-11   1230           5.344     0.519  0.241     0.121             82
2011-12   1230           5.224     0.559  0.244     0.147             82
2012-13    720           5.218     0.568  0.225     0.135             48
2013-14   1230           5.239     0.537  0.250     0.145             82
2014-15   1230           5.214     0.541  0.249     0.138             82
2015-16   1230           5.201     0.529  0.224     0.087             82
2016-17   1230           5.296     0.559  0.235     0.080             82
2017-18   1271           5.709     0.563  0.233     0.081            

## 3. What real scores look like

Every regulation goal falls into one of three kinds (2021-22 to 2025-26, where the shot data marks empty nets):

- **normal**
- **into an empty net**
- **by a team whose own goalie is pulled** (an extra attacker)

**The check:** if normal goals behave like independent Poisson draws, the share of games tied on normal goals should match what Poisson predicts. That's the Skellam distribution: the chance that two Poisson counts end up equal.

In [3]:
kinds = recent[["normal_home", "normal_away", "reg_home", "reg_away"]].sum()
total = kinds["reg_home"] + kinds["reg_away"]
normal = kinds["normal_home"] + kinds["normal_away"]
print(f"Regulation goals 2021-22 to 2025-26: {total:,}; normal {normal / total:.1%}, late (empty net or extra attacker) {1 - normal / total:.1%}")
nh, na = recent["normal_home"].mean(), recent["normal_away"].mean()
m = recent["normal_home"] - recent["normal_away"]
print(f"Normal goals per game: home {nh:.3f}, away {na:.3f}")
print(f"Games tied on normal goals: {(m == 0).mean():.3f}; Poisson (Skellam) predicts {skellam.pmf(0, nh, na):.3f}")
print(f"Spread of the normal-goal margin: {m.std():.3f}; Poisson predicts {np.sqrt(nh + na):.3f}")
fin = recent["reg_home"] - recent["reg_away"]
print(f"\nGames tied after regulation (all goals): {(fin == 0).mean():.3f}")
print("Share of games by final regulation margin vs. by normal-goal margin:")
print(pd.DataFrame({"normal goals": m.value_counts(normalize=True), "all regulation goals": fin.value_counts(normalize=True)})
      .reindex(range(-5, 6)).round(3).T.to_string())

Regulation goals 2021-22 to 2025-26: 39,431; normal 91.6%, late (empty net or extra attacker) 8.4%
Normal goals per game: home 2.851, away 2.657
Games tied on normal goals: 0.173; Poisson (Skellam) predicts 0.174
Spread of the normal-goal margin: 2.326; Poisson predicts 2.347

Games tied after regulation (all goals): 0.221
Share of games by final regulation margin vs. by normal-goal margin:
                         -5     -4     -3     -2     -1     0      1      2      3      4      5 
normal goals          0.014  0.031  0.063  0.105  0.162  0.173  0.170  0.121  0.076  0.044  0.019
all regulation goals  0.016  0.048  0.107  0.094  0.084  0.221  0.093  0.101  0.132  0.060  0.021


In [4]:
LATE = S.late_table(recent)
print("Late game, by the normal-goal margin. Chance of (leader's late goals, trailer's late goals); 1%+ shown:")
for k in range(S.MAX_MARGIN + 1):
    Lg, Tg, p = LATE[k]
    head = "tied" if k == 0 else (f"lead of {k}" if k < S.MAX_MARGIN else f"lead of {k}+")
    print(f"  {head:10s} " + ", ".join(f"({a},{b}) {q:.1%}" for a, b, q in zip(Lg, Tg, p) if q >= 0.01))
print(f"Late goals per game: {LATE['extra_goals']:.3f}")
reg_pg = results.groupby("season")["reg_goals"].mean()

def normal_total(seasons):
    """Normal goals per game going into a season: regulation goals per game in `seasons` minus the late goals."""
    return float(reg_pg.reindex(seasons).dropna().mean() - LATE["extra_goals"])

print(f"Stretch (how much the late game widens the expected margin): {S.stretch(LATE, normal_total([2023, 2024, 2025])):.3f}")

Late game, by the normal-goal margin. Chance of (leader's late goals, trailer's late goals); 1%+ shown:
  tied       (0,0) 95.2%, (0,1) 1.4%, (1,0) 1.4%
  lead of 1  (0,0) 42.7%, (0,1) 14.7%, (1,0) 33.3%, (1,1) 1.7%, (2,0) 6.9%
  lead of 2  (0,0) 27.9%, (0,1) 10.5%, (0,2) 2.6%, (1,0) 51.6%, (1,1) 4.3%, (2,0) 1.3%, (2,1) 1.6%
  lead of 3  (0,0) 61.2%, (0,1) 4.4%, (1,0) 25.0%, (1,1) 6.9%
  lead of 4+ (0,0) 95.0%, (1,0) 3.2%
Late goals per game: 0.504
Stretch (how much the late game widens the expected margin): 1.070


**Reading it:** normal goals are Poisson; the late game is what makes real hockey scores look the way they do:

- **More overtime:** extra-attacker goals turn one-goal games into ties.
- **Fewer one-goal finishes:** empty-net goals turn one-goal leads into two-goal wins.

The simulator draws from this table instead of assuming it.

**Said plainly:** the late table is measured on 2021-22 to 2025-26 (test seasons), because older seasons don't mark empty nets. It's a pattern of how games end, not of team strength.

## 4. Projections and pools for past seasons

Exactly as in notebook 23's backtest:

- **Projections:** notebook 18's setup for skaters (with the check that it reproduces notebook 18's test error) and notebook 15's for goalies. Rookies get the prior alone, or the position average.
- **Coach score:** usage from earlier seasons only, then the step 1 lineup rules.

**Two pools** for each team-season:

- **Main, 1+ games:** every player who played for the team that season. A star who missed most of the season stays in, which matches the simulator's "full health".
- **Sensitivity, 20+ games:** notebook 23's pool. It quietly uses in-season knowledge: long-injured stars drop out, and it knows which rookies stuck.

**The leak that remains, even in the main pool:**

- A traded player counts fully for his final team (MoneyPuck has one row per player-season), deadline moves included.
- The pool knows who played at all, for example which call-ups got NHL games.

In [5]:
f_train, m_train = training_nhle(careers, TRAIN_END)
nhle_train = best_nhle(careers, f_train, m_train, rule="last2")
pairs = pd.concat([candidates(history, TARGETS + [2026]),
                   skaters[skaters["season"].isin(TARGETS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
feat = skater_features(pairs, history, bios, profiles, nhle_train)
model = fit_prior(fit_rows(history, feat, TRAIN), PRIOR_INPUTS)
cand = candidates(history, TARGETS).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
priors = cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())
rows = []
for t in TARGETS:
    curve = aging_curve(history, bios, exclude_season=t)[1]
    ps = project(skaters, bios, t, SK_WEIGHTS, SK_REG, AGING, curve, priors=priors, prior_reg=1500)
    pg = project(goalies, bios, t, G_WEIGHTS, G_REG, AGING, curve)
    rows.append(pd.concat([ps, pg])[["proj_war_60"]].assign(season=t).rename_axis("playerId").reset_index())
bt_proj = pd.concat(rows, ignore_index=True)
b = backtest(skaters, TEST, SK_WEIGHTS, SK_REG, 300, bios, aging=AGING, priors=priors, prior_reg=1500)
b = b.merge(feat[["playerId", "season", "age"]], on=["playerId", "season"], how="left")
k = b["age"] > 0
err = np.sqrt(np.average((b["actual"][k] - b["pred"][k]) ** 2, weights=b["toi"][k]))
assert round(err, 4) == 0.2665, err
print(f"Past-season projections rebuilt; test-season error {err:.4f} = notebook 18's")

Past-season projections rebuilt; test-season error 0.2665 = notebook 18's


In [6]:
faceoffs = L.faceoff_rates([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"])
usage = L.usage_table(skater_war, goalie_war, faceoffs)
exp = experience(history, profiles, history[["playerId", "season"]])

def make_pool(t, min_skater, min_goalie):
    """Players with min_skater+ games (goalies min_goalie+) for a team in season t, with projection and usage."""
    u = usage[usage["season"] == t]
    keep = np.where(u["group"] == "G", u["games_played"] >= min_goalie, u["games_played"] >= min_skater)
    p = (u[keep][["playerId", "name", "team", "position", "group", "games_played", "usage"]]
         .rename(columns={"games_played": "gp", "usage": "actual"}).reset_index(drop=True))
    p["proj_war_60"] = p["playerId"].map(bt_proj[bt_proj["season"] == t].set_index("playerId")["proj_war_60"])
    miss = p["proj_war_60"].isna() & (p["group"] != "G")
    f = p.loc[miss, ["playerId", "group"]].merge(feat[feat["season"] == t].drop(columns="group"), on="playerId", how="left")
    p.loc[miss, "proj_war_60"] = (apply_prior(f, model) / GOALS_PER_WIN).to_numpy()
    still = p["proj_war_60"].isna()
    p.loc[still, "proj_war_60"] = p.loc[still, "group"].map(group_average(history, t - 1) / GOALS_PER_WIN)
    p["usage_last"] = L.usage_by_group(usage, p, t, LOOKBACK)
    p["natural_center"] = L.natural_center(usage, p["playerId"], p["position"], t) & (p["group"] == "F")
    g = p["group"] == "G"
    p.loc[g, "actual"] = p.loc[g, "gp"] / p[g].groupby("team")["gp"].transform("sum")
    p["season"] = t
    return p

ctx = {}
for t in TARGETS:
    seasons = [t - 3, t - 2, t - 1]
    ctx[t] = {"defaults": L.rookie_defaults(usage, exp, seasons), "slots": L.slot_minutes(usage, seasons),
              "share": L.starter_share(usage, seasons)[1], "totals": L.team_minutes(usage, seasons),
              "pool 1+": make_pool(t, 1, 1), "pool 20+": make_pool(t, L.MIN_GP_POOL, 10)}
for kind in ["pool 1+", "pool 20+"]:
    allp = pd.concat([c[kind] for c in ctx.values()])
    print(f"{kind}: {len(allp):,} player-seasons, {allp.groupby(['season', 'team']).ngroups} team-seasons, "
          f"{allp.groupby(['season', 'team']).size().mean():.1f} players per team")

pool 1+: 14,099 player-seasons, 432 team-seasons, 32.6 players per team
pool 20+: 10,616 player-seasons, 432 team-seasons, 24.6 players per team


## 5. Lineup fix 1: minutes blended with a player's own ice time

Slot minutes give every team's best forward the same minutes: McDavid gets 19.3, but he really plays about 23.

**The fix:** minutes = (1 - m) x slot + m x his own recent ice time per game. Players without their own ice time keep the slot. Then each team's dressed forwards and defensemen are scaled back to the slot totals, so a game still has 180.6 forward minutes and 116.6 defense minutes.

**The test:** notebook 23's ice-time backtest (players with 20+ games for the team), for m = 0 (notebook 23) to 1 (own minutes only).

In [7]:
def rmse_toi(d):
    return float(np.sqrt(np.average((d["pred"] - d["actual"]) ** 2, weights=d["gp"])))

rows, mcd = [], []
for m in [0, 0.25, 0.5, 0.75, 1]:
    res = []
    for t, c in ctx.items():
        sc = L.coach_scores(c["pool 20+"], CHOSEN, c["defaults"])
        lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
        lu["pred"] = L.assign_minutes(lu, c["slots"], c["share"], col="raw", extras=True, own_weight=m)
        res.append(lu)
    r = pd.concat(res)
    r["period"] = np.where(r["season"] <= TRAIN_END, "train", "test")
    for per in ["test", "train"]:
        x = r[r["period"] == per]
        rows.append({"m": m, "period": per, "F error": rmse_toi(x[x["group"] == "F"]), "D error": rmse_toi(x[x["group"] == "D"])})
    x = r[(r["name"] == "Connor McDavid") & (r["season"] == 2025)].iloc[0]
    mcd.append({"m": m, "McDavid 2025-26 predicted": x["pred"], "actual": x["actual"]})
blend_grid = pd.DataFrame(rows)
print("Ice-time error (minutes per game), by blend weight m:")
print(blend_grid.round(4).to_string(index=False))
assert abs(blend_grid.loc[(blend_grid["m"] == 0) & (blend_grid["period"] == "test"), "F error"].iloc[0] - 1.840958) < 1e-6  # = notebook 23
OWN = {}
for g in ["F", "D"]:
    te = blend_grid[blend_grid["period"] == "test"].set_index("m")[f"{g} error"]
    tr = blend_grid[blend_grid["period"] == "train"].set_index("m")[f"{g} error"]
    OWN[g] = float(te.idxmin())
    print(f"{NAMES[g]}: m = {OWN[g]} (test error {te.min():.4f} vs. {te.loc[0]:.4f} with slot minutes). "
          f"Best on training seasons: m = {tr.idxmin()} ({'agrees' if tr.idxmin() == OWN[g] else 'differs'})")
print("\n" + pd.DataFrame(mcd).round(2).to_string(index=False))

Ice-time error (minutes per game), by blend weight m:
   m period  F error  D error
0.00   test   1.8410   2.1879
0.00  train   2.0812   2.3698
0.25   test   1.7548   2.0831
0.25  train   1.9842   2.2744
0.50   test   1.7045   2.0263
0.50  train   1.9280   2.2278
0.75   test   1.6932   2.0221
0.75  train   1.9164   2.2340
1.00   test   1.7222   2.0714
1.00  train   1.9508   2.2942
Forwards: m = 0.75 (test error 1.6932 vs. 1.8410 with slot minutes). Best on training seasons: m = 0.75 (agrees)
Defensemen: m = 0.75 (test error 2.0221 vs. 2.1879 with slot minutes). Best on training seasons: m = 0.5 (differs)

   m  McDavid 2025-26 predicted  actual
0.00                      20.12   22.99
0.25                      20.57   22.99
0.50                      21.02   22.99
0.75                      21.46   22.99
1.00                      21.91   22.99


## 6. Preseason lineup strength for every past team-season

Each team's lineup (step 1 rules, the chosen minutes, slots scaled to each era's real minutes per game) gives two numbers:

- **skater WAR per game** = the sum over the 18 dressed skaters of projected WAR/60 x minutes per game / 60;
- **goalie WAR per game** = the same for the two goalies (the starter plays his share of games).

Team codes are matched to the NHL's (L.A to LAK, Phoenix's PHX, and so on).

In [8]:
lineups = {}
for kind in ["pool 1+", "pool 20+"]:
    for t, c in ctx.items():
        sc = L.coach_scores(c[kind], CHOSEN, c["defaults"])
        lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
        lu["team"] = [nhl_code(x, t) for x in lu["team"]]
        lu["minutes_per_game"] = L.assign_minutes(lu, L.scaled_slots(c["slots"], c["totals"]), c["share"], own_weight=OWN)
        lu["age"] = add_age(lu[["playerId"]].assign(season=t), bios)["age"].to_numpy()
        lu["draft_overall"] = lu["playerId"].map(bios.set_index("playerId")["draft_overall"])
        lineups[(kind, t)] = lu.reset_index(drop=True)

def strength_table(kind):
    out = []
    for t in TARGETS:
        teams, sk, gk = S.team_strength(lineups[(kind, t)])
        out.append(pd.DataFrame({"season": t, "team": teams, "skater": sk[0], "goalie": gk[0]}))
    return pd.concat(out, ignore_index=True)

STRENGTH = {kind: strength_table(kind) for kind in ["pool 1+", "pool 20+"]}

def with_strength(kind):
    """Every game in a target season with ds and dg (home minus away skater and goalie WAR per game)."""
    s = STRENGTH[kind].set_index(["season", "team"])
    g = results[results["season"].isin(TARGETS)].copy()
    h = s.reindex(pd.MultiIndex.from_arrays([g["season"], g["home"]]))
    a = s.reindex(pd.MultiIndex.from_arrays([g["season"], g["away"]]))
    assert h.notna().all().all() and a.notna().all().all()          # every team in every game has a lineup
    g["ds"] = h["skater"].to_numpy() - a["skater"].to_numpy()
    g["dg"] = h["goalie"].to_numpy() - a["goalie"].to_numpy()
    return g

GAMES = {kind: with_strength(kind) for kind in ["pool 1+", "pool 20+"]}
x = STRENGTH["pool 1+"].groupby("season")[["skater", "goalie"]].agg(["mean", "std"])
x.index = x.index.map(label)
print("Team WAR per game (main pool): league mean and spread across teams")
print(x.round(4).to_string())
m = STRENGTH["pool 1+"].merge(real, on=["season", "team"])
corr = m.groupby("season").apply(lambda d: (d["points"] / d["gp"]).corr(d["skater"] + d["goalie"]), include_groups=False)
print("\nCorrelation of preseason strength with real points per game, by season:")
print(corr.rename(index=label).round(2).to_string())

Team WAR per game (main pool): league mean and spread across teams
         skater          goalie        
           mean     std    mean     std
season                                 
2009-10  0.1833  0.0305  0.0479  0.0061
2010-11  0.1873  0.0346  0.0488  0.0086
2011-12  0.1937  0.0290  0.0504  0.0089
2013-14  0.1981  0.0296  0.0516  0.0059
2014-15  0.1967  0.0318  0.0519  0.0073
2015-16  0.2102  0.0284  0.0518  0.0067
2016-17  0.2172  0.0235  0.0522  0.0087
2017-18  0.2072  0.0249  0.0508  0.0071
2018-19  0.2005  0.0319  0.0501  0.0074
2021-22  0.2197  0.0354  0.0508  0.0065
2022-23  0.2138  0.0410  0.0510  0.0073
2023-24  0.2047  0.0353  0.0513  0.0099
2024-25  0.1998  0.0371  0.0512  0.0083
2025-26  0.2098  0.0346  0.0509  0.0084

Correlation of preseason strength with real points per game, by season:
season
2009-10    0.30
2010-11    0.47
2011-12    0.11
2013-14    0.12
2014-15    0.29
2015-16    0.35
2016-17    0.42
2017-18    0.13
2018-19    0.65
2021-22    0.50
2022-23    0.

## 7. Fitting the game model on preseason lineup WAR

**The fit:** least squares of each game's regulation goal difference (0 if it went to overtime) on the home team's edge in skater WAR per game (`ds`) and goalie WAR per game (`dg`). The constant is home ice. Dividing by the late-game stretch gives the normal-goal coefficients the simulator uses.

**Which seasons each fit uses:**

- **Training seasons:** fit on all training seasons (in-sample, as training should be).
- **Test seasons:** each one is fit on **every target season before it** (2018-19 on 2009-10 to 2017-18; 2025-26 on everything through 2024-25). That uses only what was known before the season.

**Why not training seasons only?** The table below shows the relationship is clearly weaker in the training era, so a training-only fit would undersell the test seasons. That fit is shown too.

In [9]:
def fit_on(kind, seasons):
    g = GAMES[kind]
    return S.fit_game_model(g[g["season"].isin(seasons)], LATE, normal_total(seasons))

def fit_seasons(t):
    return TRAIN if t in TRAIN else [s for s in TARGETS if s < t]

rows = []
for name, seasons in [("training seasons", TRAIN), ("test seasons", TEST), ("all seasons", TARGETS)]:
    f = fit_on("pool 1+", seasons)
    rows.append({"fit on": name, "games": f["n_games"], "home (goals)": f["c0"], "skater: goals per WAR": f["cs"],
                 "SE": f["se_cs"], "goalie: goals per WAR": f["cg"], "SE ": f["se_cg"], "stretch": f["stretch"],
                 "OT tilt": f["ot_b1"], "home wins OT": f["ot_home_rate"]})
fits = pd.DataFrame(rows)
print("Regulation goal difference per game, per WAR per game of edge (main pool):")
print(fits.round(3).to_string(index=False))
by = []
for t in TARGETS:
    f = fit_on("pool 1+", [t])
    by.append({"season": label(t), "skater": f["cs"], "goalie": f["cg"]})
print("\nOne season at a time (noisy, but the trend is the point):")
print(pd.DataFrame(by).set_index("season").T.round(1).to_string())
print("\nNotebook 21, on ACTUAL team WAR: 4.66 goals per skater WAR, 6.17 per goalie WAR.")

Regulation goal difference per game, per WAR per game of edge (main pool):
          fit on  games  home (goals)  skater: goals per WAR    SE  goalie: goals per WAR   SE   stretch  OT tilt  home wins OT
training seasons   9881         0.272                  3.918 0.559                  6.133 2.175    1.079    0.664         0.512
    test seasons   7830         0.227                  8.347 0.568                 10.646 2.559    1.070    0.291         0.516
     all seasons  17711         0.252                  6.367 0.395                  9.057 1.656    1.075    0.391         0.513

One season at a time (noisy, but the trend is the point):
season  2009-10  2010-11  2011-12  2013-14  2014-15  2015-16  2016-17  2017-18  2018-19  2021-22  2022-23  2023-24  2024-25  2025-26
skater      5.5      4.4     -0.2      2.8      4.7      5.5      9.5      0.7      9.7      9.1     10.3     10.2      4.3      6.3
goalie      0.1     15.2      9.7     30.0      1.6     -3.3      2.1      4.0     -5.4 

In [10]:
season_fits = {(kind, t): fit_on(kind, fit_seasons(t)) for kind in ["pool 1+", "pool 20+"] for t in TARGETS}
rows = [{"season": label(t), "fit on": "training seasons" if t in TRAIN else f"{label(TARGETS[0])} to {label(fit_seasons(t)[-1])}",
         "a_s (normal goals per skater WAR)": season_fits[("pool 1+", t)]["a_s"],
         "a_g (per goalie WAR)": season_fits[("pool 1+", t)]["a_g"], "home": season_fits[("pool 1+", t)]["home"]}
        for t in [TRAIN[0]] + TEST]
print("Coefficients the backtest uses (main pool):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

Coefficients the backtest uses (main pool):
 season             fit on  a_s (normal goals per skater WAR)  a_g (per goalie WAR)  home
2009-10   training seasons                              3.632                 5.684 0.252
2018-19 2009-10 to 2017-18                              3.632                 5.684 0.252
2021-22 2009-10 to 2018-19                              4.272                 5.029 0.257
2022-23 2009-10 to 2021-22                              4.945                 6.262 0.253
2023-24 2009-10 to 2022-23                              5.702                 6.931 0.244
2024-25 2009-10 to 2023-24                              6.164                 9.571 0.239
2025-26 2009-10 to 2024-25                              5.930                 9.274 0.243


**Reading the coefficients:** notebook 21's 4.66 and 6.17 are per *actual* WAR over a season. These are per *projected* WAR, game by game.

- **If the projections were calibrated,** the two would be about the same. A forecast that's right on average converts one-for-one.
- **In the training era they're lower.** There, preseason projections tracked team results less well: our data starts in 2008-09, so the early projections have little history behind them.
- **In recent seasons they're higher.**
- **Goalie coefficients are noisy** (large SE), because the spread of goalie WAR per game between teams is small.

## 8. Do simulated games look like real games?

Every test-season game, simulated 200 times with its two teams' preseason strengths, using the coefficients that season uses (no player randomness). The comparison is with what really happened.

In [11]:
rng = np.random.default_rng(SEED)
sim_rows = []
for t in TEST:
    g = GAMES["pool 1+"][GAMES["pool 1+"]["season"] == t]
    sett = S.game_settings(season_fits[("pool 1+", t)], LATE, normal_total([t - 3, t - 2, t - 1]))
    teams = np.array(sorted(set(g["home"]) | set(g["away"])))
    st = STRENGTH["pool 1+"][STRENGTH["pool 1+"]["season"] == t].set_index("team").reindex(teams)
    sk = np.repeat(st[["skater"]].to_numpy().T, 200, axis=0)
    gk = np.repeat(st[["goalie"]].to_numpy().T, 200, axis=0)
    hg, ag, ot, hw = S.simulate_games(np.searchsorted(teams, g["home"]), np.searchsorted(teams, g["away"]), sk, gk, sett, rng)
    reg_margin = np.where(ot, 0, hg - ag)
    sim_rows.append(pd.DataFrame({"reg_margin": reg_margin.ravel(), "ot": ot.ravel(), "home_win": hw.ravel(),
                                  "goals": (hg + ag - ot).ravel()}))
sim = pd.concat(sim_rows)
rt = results[results["season"].isin(TEST)]
print(pd.DataFrame({"real": [rt["ot"].mean(), rt["home_win"].mean(), rt["reg_goals"].mean()],
                    "simulated": [sim["ot"].mean(), sim["home_win"].mean(), sim["goals"].mean()]},
                   index=["went to OT/SO", "home team won", "regulation goals per game"]).round(3).to_string())
print("\nRegulation margin (home - away; 0 = went to OT/SO), share of games:")
print(pd.DataFrame({"real": rt["reg_margin"].value_counts(normalize=True), "simulated": sim["reg_margin"].value_counts(normalize=True)})
      .reindex(range(-5, 6)).round(3).T.to_string())

                            real  simulated
went to OT/SO              0.221      0.221
home team won              0.537      0.543
regulation goals per game  5.981      5.836

Regulation margin (home - away; 0 = went to OT/SO), share of games:
reg_margin     -5     -4     -3     -2     -1     0      1      2      3      4      5 
real        0.016  0.047  0.107  0.092  0.087  0.221  0.093  0.102  0.132  0.060  0.022
simulated   0.014  0.047  0.110  0.090  0.081  0.221  0.090  0.103  0.136  0.066  0.022


## 9. The standings backtest (the main test)

For every target season, every team's season is simulated **1,000 times** on that season's **real schedule**. The inputs are its preseason lineup, that season's coefficients, and scoring from the three seasons before.

**Player randomness,** three versions:

- **(i) none:** every player plays exactly his projection.
- **(ii) full:** notebook 17's draw, one per player per simulated season. His season can be better or worse than projected, as real seasons are.
- **(iii) non-luck:** the same draw minus its sampling-luck share. The idea is that the game model already adds game-to-game luck.

**One correction to notebook 17's pools:** they're **centered** (each pool's average removed). Notebook 17 measured misses against notebook 15's projection, so its pools lean: young top picks beat it, players 33+ fell short. Notebook 18's prior already moved the projection that way, so drawing the lean again would count it twice. The uncentered version is shown as a comparison row. This was decided on the reasoning, not picked from the results.

**Two simple rules to beat:**

- **(a) every team at the league average** (last season's points per game x games);
- **(b) last season's points per game, pulled one-third toward average,** x games. Expansion teams (VGK 2017-18, SEA 2021-22) get the average; 2012-13 and 2020-21 count per game; relocated teams carry their history (ATL to WPG, PHX to ARI, ARI to UTA).

**Measures:**

- **accuracy:** error (RMSE) in points and the within-season correlation with real points;
- **spread:** the spread (SD) of team points across the league, real vs. simulated, and how often a team's real total lands inside its simulated 80% and 50% ranges (targets 80% and 50%).

In [12]:
INPUTS = {c: S.draw_inputs(pd.read_csv(DATA / "dev_residuals.csv"), pd.read_csv(DATA / "dev_ranges.csv"), center_pools=c)
          for c in [True, False]}
RUNS = [("pool 1+", "none"), ("pool 1+", "full"), ("pool 1+", "non-luck"), ("pool 1+", "full, uncentered pools"),
        ("pool 20+", "none"), ("pool 20+", "full"), ("pool 20+", "non-luck")]

def run_season(kind, parts, t, n=N_SIMS):
    lu = lineups[(kind, t)]
    sched = results[results["season"] == t][["home", "away"]]
    gp = pd.concat([sched["home"], sched["away"]]).value_counts()
    minutes = lu["minutes_per_game"] * lu["team"].map(gp).to_numpy()            # his season's minutes
    inputs = INPUTS["uncentered" not in parts]
    buckets = S.draw_buckets(lu["group"], lu["age"], lu["draft_overall"], inputs)
    draws = S.player_draws(lu["group"].to_numpy(), buckets, minutes, inputs, parts.split(",")[0], n, seed=t)
    sett = S.game_settings(season_fits[(kind, t)], LATE, normal_total([t - 3, t - 2, t - 1]))
    table, pts = S.simulate_season(lu, sched, sett, n, seed=t, draws=draws)
    table["season"], table["pool"], table["randomness"] = t, kind, parts
    table["sim_sd"] = pts.std(axis=1).mean()                                   # spread across teams, per simulated season
    return table

start = time.time()
bt = pd.concat([run_season(kind, parts, t) for kind, parts in RUNS for t in TARGETS], ignore_index=True)
bt = bt.merge(real[["season", "team", "gp", "points"]].rename(columns={"points": "real"}), on=["season", "team"])
bt["period"] = np.where(bt["season"] <= TRAIN_END, "train", "test")
print(f"{len(RUNS)} versions x {len(TARGETS)} seasons x {N_SIMS:,} simulated seasons in {time.time() - start:.0f} seconds")

7 versions x 14 seasons x 1,000 simulated seasons in 35 seconds


In [13]:
real["ppg"] = real["points"] / real["gp"]
avg_prev = real.groupby("season")["ppg"].mean()
avg_prev.index = avg_prev.index + 1
prev = real[["season", "team", "ppg"]].copy()
prev["season"] += 1
MOVED = {("ATL", 2011): "WPG", ("PHX", 2014): "ARI", ("ARI", 2024): "UTA"}
prev["team"] = [MOVED.get((tm, s), tm) for tm, s in zip(prev["team"], prev["season"])]
base = bt[bt["pool"] == "pool 1+"].drop_duplicates(["season", "team"])[["season", "team", "gp", "real", "period"]].copy()
base = base.merge(prev, on=["season", "team"], how="left")
base["avg"] = base["season"].map(avg_prev)
print("No last season (league average used):", base[base["ppg"].isna()][["season", "team"]].values.tolist())
base["(a) league average"] = base["avg"] * base["gp"]
base["(b) last season, 1/3 to average"] = np.where(base["ppg"].isna(), base["avg"], 2 / 3 * base["ppg"] + 1 / 3 * base["avg"]) * base["gp"]

def accuracy(pred, d):
    """Error in points, the average within-season correlation (none when every team gets the same prediction), bias."""
    e = pred - d["real"]
    corr = d.assign(p=pred).groupby("season").apply(
        lambda x: x["p"].corr(x["real"]) if x["p"].std() > 1e-6 else np.nan, include_groups=False).mean()
    return {"error (RMSE, points)": float(np.sqrt((e ** 2).mean())), "correlation": float(corr), "bias": float(e.mean())}

rows = []
for per in ["train", "test"]:
    d = base[base["period"] == per]
    for col in ["(a) league average", "(b) last season, 1/3 to average"]:
        acc = accuracy(d[col], d)
        if col.startswith("(a)"):
            acc["correlation"] = np.nan          # every team gets the same points (games barely differ): no correlation
        rows.append({"period": per, "model": col, **acc})
    for kind, parts in RUNS:
        x = bt[(bt["pool"] == kind) & (bt["randomness"] == parts) & (bt["period"] == per)]
        rows.append({"period": per, "model": f"simulator, {kind}, {parts}", **accuracy(x["points"], x),
                     "SD real": x.groupby("season")["real"].std().mean(), "SD simulated": x.groupby("season")["sim_sd"].first().mean(),
                     "inside 80% range": ((x["real"] >= x["p10"]) & (x["real"] <= x["p90"])).mean(),
                     "inside 50% range": ((x["real"] >= x["p25"]) & (x["real"] <= x["p75"])).mean()})
score = pd.DataFrame(rows)
for per in ["test", "train"]:
    print(f"{per.capitalize()} seasons:")
    print(score[score["period"] == per].drop(columns="period").round(3).to_string(index=False))
    print()

No last season (league average used): [[2017, 'VGK'], [2021, 'SEA']]
Test seasons:
                                     model  error (RMSE, points)  correlation   bias  SD real  SD simulated  inside 80% range  inside 50% range
                        (a) league average                16.404          NaN -0.033      NaN           NaN               NaN               NaN
           (b) last season, 1/3 to average                13.561        0.552 -0.033      NaN           NaN               NaN               NaN
                  simulator, pool 1+, none                14.366        0.516  0.017   16.414        10.218             0.555             0.304
                  simulator, pool 1+, full                14.381        0.513 -0.148   16.414        14.024             0.764             0.466
              simulator, pool 1+, non-luck                14.337        0.519  0.005   16.414        10.659             0.592             0.319
simulator, pool 1+, full, uncentered pools           

In [14]:
# Player randomness: spread first (inside the 80% and 50% ranges, closest to 80% and 50%), then accuracy
main = score[score["model"].str.startswith("simulator, pool 1+") & ~score["model"].str.contains("uncentered")].copy()
main["spread miss"] = (main["inside 80% range"] - 0.8).abs() + (main["inside 50% range"] - 0.5).abs()
pick = {per: main[main["period"] == per].sort_values(["spread miss", "error (RMSE, points)"])["model"].iloc[0] for per in ["test", "train"]}
RANDOMNESS = pick["test"].split(", ")[-1]
print(f"Best spread on the test seasons: {pick['test']}; on the training seasons: {pick['train']} "
      f"({'agrees' if pick['test'] == pick['train'] else 'differs'})")
print(f"Player randomness used from here on: {RANDOMNESS}")

t_b = score[(score["period"] == "test") & (score["model"] == "(b) last season, 1/3 to average")].iloc[0]
t_m = score[(score["period"] == "test") & (score["model"] == f"simulator, pool 1+, {RANDOMNESS}")].iloc[0]
t_20 = score[(score["period"] == "test") & (score["model"] == f"simulator, pool 20+, {RANDOMNESS}")].iloc[0]
tr_b = score[(score["period"] == "train") & (score["model"] == "(b) last season, 1/3 to average")].iloc[0]
tr_m = score[(score["period"] == "train") & (score["model"] == f"simulator, pool 1+, {RANDOMNESS}")].iloc[0]
verdict = lambda m, b: "BEATS" if m["error (RMSE, points)"] < b["error (RMSE, points)"] else "does NOT beat"
print(f"\nVERDICT (test seasons, main pool): the simulator {verdict(t_m, t_b)} rule (b): error {t_m['error (RMSE, points)']:.2f} "
      f"vs. {t_b['error (RMSE, points)']:.2f} points, correlation {t_m['correlation']:.3f} vs. {t_b['correlation']:.3f}.")
print(f"Training seasons: {verdict(tr_m, tr_b)} (b): {tr_m['error (RMSE, points)']:.2f} vs. {tr_b['error (RMSE, points)']:.2f}.")
print(f"With the 20+ game pool (uses in-season knowledge): {verdict(t_20, t_b)} (b) on test: "
      f"{t_20['error (RMSE, points)']:.2f} vs. {t_b['error (RMSE, points)']:.2f}.")

Best spread on the test seasons: simulator, pool 1+, full; on the training seasons: simulator, pool 1+, full (agrees)
Player randomness used from here on: full

VERDICT (test seasons, main pool): the simulator does NOT beat rule (b): error 14.38 vs. 13.56 points, correlation 0.513 vs. 0.552.
Training seasons: does NOT beat (b): 13.42 vs. 12.82.
With the 20+ game pool (uses in-season knowledge): does NOT beat (b) on test: 13.76 vs. 13.56.


**Why doesn't it beat (b), and what would?** Two diagnostics, using the chosen randomness on the main pool.

- **Calibration slope:** regress real points on simulated average points, both centered within each season.
  - A slope of 1 means the simulated gaps between teams are the right size.
  - Above 1, real gaps are bigger than simulated, so the simulator is too timid.
- **A blend with (b):** w x simulator + (1 - w) x rule (b), with w fit on the **training seasons only**, then scored on the test seasons.
  - If the blend beats (b), the simulator knows something last season's points don't (and vice versa).
  - It's a diagnostic for the next step, not part of the simulator.

In [15]:
x = bt[(bt["pool"] == "pool 1+") & (bt["randomness"] == RANDOMNESS)].merge(
    base[["season", "team", "(b) last season, 1/3 to average"]], on=["season", "team"])
x = x.rename(columns={"(b) last season, 1/3 to average": "rule_b"})
for per in ["train", "test"]:
    d = x[x["period"] == per]
    slope = np.polyfit(d["points"] - d.groupby("season")["points"].transform("mean"),
                       d["real"] - d.groupby("season")["real"].transform("mean"), 1)[0]
    print(f"{per.capitalize()} seasons: calibration slope {slope:.2f}; spread of simulated averages across teams "
          f"{d.groupby('season')['points'].std().mean():.1f} points vs. real {d.groupby('season')['real'].std().mean():.1f}")
tr, te = x[x["period"] == "train"], x[x["period"] == "test"]
ws = np.round(np.arange(0, 1.0001, 0.05), 2)
w = ws[int(np.argmin([np.sqrt(((v * tr["points"] + (1 - v) * tr["rule_b"] - tr["real"]) ** 2).mean()) for v in ws]))]
blend = w * te["points"] + (1 - w) * te["rule_b"]
acc = accuracy(blend, te)
print(f"\nBlend: {w:.2f} x simulator + {1 - w:.2f} x rule (b), weight fit on training seasons. On test seasons: "
      f"error {acc['error (RMSE, points)']:.2f} (rule (b) alone {t_b['error (RMSE, points)']:.2f}, simulator alone "
      f"{t_m['error (RMSE, points)']:.2f}), correlation {acc['correlation']:.3f}")

Train seasons: calibration slope 1.04; spread of simulated averages across teams 3.6 points vs. real 14.1
Test seasons: calibration slope 1.41; spread of simulated averages across teams 5.9 points vs. real 16.4

Blend: 0.40 x simulator + 0.60 x rule (b), weight fit on training seasons. On test seasons: error 13.06 (rule (b) alone 13.56, simulator alone 14.38), correlation 0.588


In [16]:
print("By season (main pool, chosen randomness):")
by = x.groupby("season").apply(lambda d: pd.Series({
    "simulator error": np.sqrt(((d["points"] - d["real"]) ** 2).mean()), "rule (b) error": np.sqrt(((d["rule_b"] - d["real"]) ** 2).mean()),
    "simulator corr": d["points"].corr(d["real"]), "rule (b) corr": d["rule_b"].corr(d["real"])}), include_groups=False)
by.index = by.index.map(label)
print(by.round(2).to_string())
te = te.assign(miss=te["real"] - te["points"])
print("\nTest-season teams the simulator missed most (real minus simulated average points):")
show = te.reindex(te["miss"].abs().sort_values(ascending=False).index).head(12)
print(show.assign(season=show["season"].map(label))[["season", "team", "real", "points", "p10", "p90", "rule_b", "miss"]].round(1).to_string(index=False))

By season (main pool, chosen randomness):
         simulator error  rule (b) error  simulator corr  rule (b) corr
season                                                                 
2009-10            12.31           11.61            0.26           0.48
2010-11            11.64           10.47            0.48           0.60
2011-12            11.71           10.35            0.14           0.51
2013-14            14.90           13.11            0.14           0.53
2014-15            15.05           13.19            0.28           0.55
2015-16            11.82           11.49            0.36           0.52
2016-17            13.95           14.48            0.42           0.33
2017-18            15.22           16.51            0.09           0.18
2018-19            11.41           11.85            0.61           0.53
2021-22            18.12           15.09            0.48           0.66
2022-23            15.46           15.69            0.63           0.56
2023-24            13.

## 10. Lineup fix 2: call-ups only when needed

A coach dresses his **NHL-level** players first (NHL roster, off the active roster, the two disputes). **AHL/ECHL players on NHL contracts** dress only for slots the NHL level can't fill, for example when someone sits, or when the NHL level has no 4th center. Within each tier, notebook 23's coach score orders players; once dressed, minutes follow the coach score.

**Said plainly: this can't be backtested.** Past pools only contain players who played NHL games, and past AHL depth isn't in our data. It's a rule of how NHL teams work, applied to 2026-27.

In [17]:
RECENT = [2023, 2024, 2025]
players = L.load_players(DATA, usage, current_projections(DATA), LOOKBACK)
DEFAULTS = L.rookie_defaults(usage, exp, RECENT)
slots26 = L.scaled_slots(L.slot_minutes(usage, RECENT), L.team_minutes(usage, RECENT))
SHARE = L.starter_share(usage, RECENT)[1]
scored26 = L.coach_scores(players[players["in_pool"]], CHOSEN, DEFAULTS)
pid = lambda name: int(players.loc[players["name"] == name, "playerId"].iloc[0])
DISPUTES = [pid("Dylan Larkin"), pid("Simon Edvinsson")]

old_rule = L.build_lineups(scored26, slots26, SHARE, own_weight=OWN)
tiered = L.build_lineups(scored26, slots26, SHARE, tiered=True, own_weight=OWN)
ahl = lambda lu: lu[lu["dressed"] & lu["status"].isin(L.AHL_STATUSES)]
print(f"AHL/ECHL players dressed on opening night: {len(ahl(old_rule))} with notebook 23's rule, {len(ahl(tiered))} with tiers")
if len(ahl(tiered)):
    print(ahl(tiered)[["team", "name", "group", "slot", "flag"]].to_string(index=False))
d_old = old_rule[(old_rule["team"] == "DET") & old_rule["dressed"] & (old_rule["group"] == "D")]
d_new = tiered[(tiered["team"] == "DET") & tiered["dressed"] & (tiered["group"] == "D")]
print("\nDetroit's dressed D, notebook 23's rule:", ", ".join(d_old["name"]))
print("Detroit's dressed D, tiers:              ", ", ".join(d_new["name"]))
both = L.build_lineups(scored26, slots26, SHARE, sit_out=DISPUTES, tiered=True, own_weight=OWN)
x = both[(both["team"] == "DET") & both["dressed"]]
print(f"\nDetroit with Larkin and Edvinsson both sitting (tiers): {len(x[x['status'].isin(L.AHL_STATUSES)])} AHL players dress; "
      f"NHL-level players available: {int(((scored26['team'] == 'DET') & ~scored26['status'].isin(L.AHL_STATUSES)).sum()) - 2}")
slot_only = L.build_lineups(scored26, slots26, SHARE, tiered=True)
mc = lambda lu: lu.loc[lu["name"] == "Connor McDavid", "minutes_per_game"].iloc[0]
print(f"McDavid's 2026-27 minutes: {mc(slot_only):.1f} with slot minutes (notebook 23), {mc(tiered):.1f} with the blend")

AHL/ECHL players dressed on opening night: 56 with notebook 23's rule, 1 with tiers
team               name group slot                                  flag
 EDM Connor Clattenburg     F  F11 center rule: dressed over Colton Dach

Detroit's dressed D, notebook 23's rule: Moritz Seider, Simon Edvinsson, Justin Faulk, Ben Chiarot, Axel Sandin-Pellikka, Anton Johansson
Detroit's dressed D, tiers:               Moritz Seider, Simon Edvinsson, Justin Faulk, Ben Chiarot, Anton Johansson, William Wallinder

Detroit with Larkin and Edvinsson both sitting (tiers): 0 AHL players dress; NHL-level players available: 26
McDavid's 2026-27 minutes: 19.3 with slot minutes (notebook 23), 21.1 with the blend


## 11. Final settings for 2026-27, and a smoke test

For 2026-27 (step 3) the game model is **refit on all seasons**, as notebook 18 did for its final prior. Scoring is 2023-24 to 2025-26's.

**The smoke test:**

- **Lineups:** 2026-27 opening night with tiers, blended minutes and the chosen randomness.
- **Schedule:** a simple balanced 84-game schedule (step 3 uses the real one).
- **Run:** one season simulated once, standings printed.

Then the run time for 1,000 and 10,000 seasons, and two checks:

- the same seed gives the same results;
- the points add up (2 per game, plus 1 for every game that went to OT/SO).

In [18]:
FINAL_FIT = fit_on("pool 1+", TARGETS)
FINAL = S.game_settings(FINAL_FIT, LATE, normal_total(RECENT))
print(f"Final game model (all seasons, {FINAL_FIT['n_games']:,} games): home {FINAL['home']:.3f} goals, "
      f"a_s {FINAL['a_s']:.3f} and a_g {FINAL['a_g']:.3f} normal goals per WAR per game, "
      f"normal scoring {FINAL['total']:.3f} goals per game, OT: home edge {FINAL['ot_b0']:.3f}, tilt {FINAL['ot_b1']:.3f}")
print(f"Settings: coach score {CHOSEN}, lookback {LOOKBACK}, minutes blend {OWN}, call-up tiers on, randomness: {RANDOMNESS}")

lu26 = tiered.copy()
lu26 = lu26.merge(current_projections(DATA)[["playerId", "age"]], on="playerId", how="left")
lu26["draft_overall"] = lu26["playerId"].map(bios.set_index("playerId")["draft_overall"])
sched26 = S.balanced_schedule(sorted(lu26["team"].unique()), L.SEASON_GAMES_2026, seed=SEED)
assert (pd.concat([sched26["home"], sched26["away"]]).value_counts() == 84).all()
inputs = INPUTS[True]
buckets26 = S.draw_buckets(lu26["group"], lu26["age"], lu26["draft_overall"], inputs)

def draws26(n, seed):
    return S.player_draws(lu26["group"].to_numpy(), buckets26, lu26["minutes_per_game"] * 84, inputs, RANDOMNESS, n, seed)

one, _ = S.simulate_season(lu26, sched26, FINAL, n_sims=1, seed=SEED, draws=draws26(1, SEED))
avg, _ = S.simulate_season(lu26, sched26, FINAL, n_sims=N_SIMS, seed=SEED, draws=draws26(N_SIMS, SEED))
one = one.sort_values("points", ascending=False).reset_index(drop=True)
one = one.astype({"points": int, "wins": int, "ot_losses": int, "goal_diff": int})
one = one.merge(avg[["team", "points", "p10", "p90"]].rename(columns={"points": "avg of 1,000", "p10": "10th", "p90": "90th"}),
                on="team")
one.index += 1
print("\nSmoke test: one simulated 2026-27 season (balanced schedule, not the real one), next to the average of 1,000:")
print(one[["team", "points", "wins", "ot_losses", "goal_diff", "avg of 1,000", "10th", "90th"]].round(1).to_string())
print("One season is noisy on purpose: the 10th-90th range is what a single season can look like.")

Final game model (all seasons, 17,711 games): home 0.234 goals, a_s 5.925 and a_g 8.427 normal goals per WAR per game, normal scoring 5.456 goals per game, OT: home edge 0.054, tilt 0.391
Settings: coach score {'F': (0.75, 'R1'), 'D': (0.75, 'R1'), 'G': (0.5, 'R1')}, lookback {'F': 2, 'D': 1, 'G': 2}, minutes blend {'F': 0.75, 'D': 0.75}, call-up tiers on, randomness: full

Smoke test: one simulated 2026-27 season (balanced schedule, not the real one), next to the average of 1,000:
   team  points  wins  ot_losses  goal_diff  avg of 1,000  10th   90th
1   DAL     117    54          9         51         102.5  85.0  120.0
2   COL     114    52         10         47          99.2  81.0  116.0
3   WSH     113    53          7         85         104.6  86.0  122.0
4   MIN     113    53          7         58          93.2  75.0  110.0
5   MTL     106    48         10         48          99.8  83.0  117.0
6   CHI     106    50          6         30          89.4  70.0  107.0
7   BUF     105 

In [19]:
a, pa = S.simulate_season(lu26, sched26, FINAL, n_sims=200, seed=7, draws=draws26(200, 7))
b, pb = S.simulate_season(lu26, sched26, FINAL, n_sims=200, seed=7, draws=draws26(200, 7))
assert np.array_equal(pa, pb)
ot_share = a["ot_losses"].sum() / len(sched26)                  # every OT/SO game has exactly one OT loser
assert abs(pa.sum(axis=1).mean() - (2 * len(sched26) + a["ot_losses"].sum())) < 1e-9
assert abs(a["wins"].sum() - len(sched26)) < 1e-9                 # one winner per game
print(f"Same seed, same results: yes. Points per game = 2 + the OT/SO share: yes "
      f"({pa.sum(axis=1).mean() / len(sched26):.3f} = 2 + {ot_share:.3f})")

for n in [1000, 10000]:
    start = time.time()
    dr = draws26(n, 11)
    t_draw = time.time() - start
    S.simulate_season(lu26, sched26, FINAL, n_sims=n, seed=11, draws=dr)
    print(f"{n:,} simulated 84-game seasons (1,344 games each): {time.time() - start:.1f} seconds "
          f"({t_draw:.1f} of them drawing player seasons)")

Same seed, same results: yes. Points per game = 2 + the OT/SO share: yes (2.214 = 2 + 0.214)
1,000 simulated 84-game seasons (1,344 games each): 0.4 seconds (0.1 of them drawing player seasons)
10,000 simulated 84-game seasons (1,344 games each): 4.6 seconds (0.9 of them drawing player seasons)


## Takeaways
- **The verdict, plainly: on its own, the simulator does not beat rule (b)** (last season's points pulled one-third toward average).
  - **Test seasons:** error 14.38 points vs. 13.56, correlation 0.513 vs. 0.552.
  - **Training seasons:** 13.42 vs. 12.82.
  - **With the 20+ game pool,** which quietly uses in-season knowledge, it still doesn't beat it (13.76 vs. 13.56). So it isn't only the fairer pool that holds it back.
- **But it knows something rule (b) doesn't.** Blending 40% simulator with 60% rule (b), the weight fit on the training seasons only, beats both on the test seasons: error 13.06, correlation 0.588. That makes a team-level "last season" ingredient a good candidate for step 3. It's your call, not built in yet.
- **Why it falls short:**
  - **Preseason projections tracked team results weakly before 2018-19.** Within-season correlation with real points was 0.11 to 0.47, vs. 0.31 to 0.70 since. Our data starts in 2008-09, so early projections have little history.
  - **That makes the coefficients climb over time.** Regulation goals per skater WAR of edge are 3.9 (SE 0.56) on the training seasons and 8.3 on the test seasons; 6.4 on all.
  - **The simulator is too timid.** Each test season is fit on every season before it (a_s rises from 3.6 to 6.2). That still undersells recent seasons: the calibration slope is 1.41, so real gaps between teams are 1.4x the simulated ones.
  - **Notebook 21's numbers on actual WAR** (4.66 skater, 6.17 goalie) sit in between.
- **The game model reproduces real games:**
  - **Normal goals are Poisson** (games tied on them: 17.3% real vs. 17.4% Poisson).
  - **The late table does the rest.** It's what really happens with the goalie pulled. After a one-goal lead on normal goals, the leader adds a late goal (mostly into an empty net) 42% of the time, and the trailer ties it 15% of the time.
  - **Test seasons, real vs. simulated:**
    - overtime/shootout: 22.1% vs. 22.1%
    - home wins: 53.7% vs. 54.3%
    - the regulation-margin pattern matches, including the rarer one-goal and common three-goal finishes
    - goals per game: 5.98 vs. 5.84, because scoring comes from the three seasons before
  - **Overtime is close to a coin flip** (home 51.3%, with a small tilt by strength).
- **Player randomness: the full notebook 17 draw, with each pool centered.**
  - **Spread:** best on both test and training seasons. 76% of real totals land in the simulated 80% range and 47% in the 50% range (no draws: 56% and 30%). The simulated spread of team points is 14.0 vs. 16.4 real.
  - **Accuracy:** about the same as no draws (14.38 vs. 14.37).
  - **Centering:** notebook 17's pools lean (young top picks up, players 33+ down), and notebook 18's prior already did that, so they're centered. Uncentered, accuracy drops (14.80). That was decided on the reasoning, not picked from the results.
  - **Plainly:** the pools were built from all seasons, test seasons included (spread only, no direction).
- **Lineup fix 1, minutes:** 75% own recent ice time and 25% slot, rescaled to the team totals.
  - It cuts the ice-time error from 1.84 to 1.69 minutes for forwards and from 2.19 to 2.02 for defensemen.
  - The training seasons agree for forwards; for defensemen they prefer 50%.
  - McDavid goes from 19.3 to 21.1 minutes in 2026-27.
- **Lineup fix 2, call-up tiers:** the AHL/ECHL players dressed on opening night drop from 56 (notebook 23's rule) to 1 (Clattenburg as Edmonton's 4th center).
  - Detroit dresses Wallinder instead of Sandin-Pellikka.
  - **Plainly:** this can't be backtested, because past AHL depth isn't in our data.
- **The leaks that remain in the backtest:**
  - A traded player counts fully for his final team, deadline moves included.
  - The pool knows who played at all, for example which call-ups got NHL games.
  - The late table is measured on test seasons (older seasons don't mark empty nets).
- **The biggest test-season misses are the famous surprises:** 2022-23 Boston (135 real vs. 93 simulated), 2018-19 Tampa (128 vs. 95), 2021-22 Philadelphia (61 vs. 97), 2024-25 San Jose (52 vs. 87). Rule (b) missed most of them too.
- **For 2026-27** (refit on all 17,711 games): home ice 0.23 normal goals, 5.93 normal goals per skater WAR per game and 8.43 per goalie WAR per game, and normal scoring 5.46 a game.
  - **Smoke test**, average of 1,000 seasons on a balanced schedule:
    - Top: Washington 104.6, Dallas 102.5, Edmonton 102.0, Buffalo 101.6, Vegas 100.6
    - Bottom: Calgary 76.3, Seattle 81.8, the Islanders 83.0
    - Detroit: 88.1
  - **A single simulated season is noisy:** Edmonton finished 29th in one.
- **The engine is fast and reproducible:** 1,000 seasons of 84 games in about half a second, 10,000 in about 4 seconds. The same seed gives the same results, and the points add up (2.214 per game = 2 + the 21.4% OT/SO share).
- **Files:** past results in `data/schedule_raw/` (gitignored). No new tables saved; step 3 runs everything from `src/simulate.py`.
- **Next (step 3), with two decisions for you:**
  1. Add last season's points as a team-level ingredient (the blend that beat rule (b)).
  2. Weight recent seasons more in the coefficient fit (the test calibration slope is 1.41).

  Then the real 2026-27 schedule, trades, sit-outs and start dates.